# Estimators battle (MoM VS. MLE VS. QB VS. Ensemble)

$(\xi_1, ..., \xi_n)$ - iid samples, $\xi_i \sim \Gamma(\alpha^2, 5)$

Let's build estimators for the gamma dsitribution parameter by:
- Method of Moments (MoM)
- Maximum Likelihood Estimation (MLE)
- Quantile Based Estimations (min/max/median/quartile) (QB)
- Combination of 3 above: (0.33 MoM + 0.33 MLE + 0.34 QB)

Useful questions to answer:
- Will these estimates be unbiased?
- Will they be consistent?
- Which estimate will have the smallest variance?



### Method of Moments (MoM)
The idea behind MoM estimation is that: to find a good estimator, we should have the <b>true and sample moments match</b> as best we can.

True (theoretical) first moment in our case: $E[\xi_i]=5\alpha^2$ \
Sample moment - $\bar{\xi} = \frac{1}{n} \sum_{i=1}^{n}\xi_i$ \
match - $5\alpha^2 = \frac{1}{n} \sum_{i=1}^{n}\xi_i$ \
$\implies \alpha_{MoM} = \sqrt{\frac{\bar{\xi}}{5}} = \sqrt{\frac{1}{5n}\sum_{i=1}^{n}\xi_i} $


Will the estimate be unbiased? \
$ E[\hat{\alpha}] = E [\sqrt{\frac{\bar{\xi}}{5}}] = \frac{1}{\sqrt{5}} E[\sqrt{\bar{\xi}}] < $ (by Jensen for strictly concave func.) $< \frac{1}{\sqrt{5}} \sqrt{E[\bar{\xi}]} = \frac{1}{\sqrt{5}} \sqrt{5 \alpha^2} = \alpha$ \
So, $ E[\hat{\alpha}] < \alpha \implies  \alpha_{MoM} $ is biased. 

Will the estimate be consistent? \
Yes, this implies from the law of large numbers.


In [46]:
import numpy as np
np.random.seed(666)

alpha     = 4
biases    = []
variances = []

for N in [10**2, 10**3, 10**4, 10**5, 10**6, 10**7]:
    alpha_mom_list = []
    for _ in range(100):
        samples = np.random.gamma(alpha*alpha, 5, size=N)
        alpha_mom = np.sqrt(np.mean(samples)/5)
        alpha_mom_list.append(alpha_mom)

    bias = np.mean(alpha_mom_list) - alpha
    biases.append(bias)

    variance = np.var(alpha_mom_list)
    variances.append(variance)


biases, variances

([np.float64(-0.010698996914846326),
  np.float64(-0.00024699856109000606),
  np.float64(0.0002672434052115591),
  np.float64(7.068445363245246e-05),
  np.float64(-3.755396513049902e-05),
  np.float64(-2.804393021493823e-05)],
 [np.float64(0.0024920780172938153),
  np.float64(0.0002744988641996945),
  np.float64(3.03702604783875e-05),
  np.float64(2.4035976313309003e-06),
  np.float64(2.8477702145455836e-07),
  np.float64(2.899766269144772e-08)])

As we can see from the simulations, asymptotically (as N increases), the estimate becomes less biased and more consistent.

### Maximum Likelihood Estimation (MLE)
The idea behind ML estimation is contained in its title: find an estimation that will <b>maximize likelihood</b> to observe such data as given.

$f_{\xi}(x;\alpha^2)=\frac{x^{\alpha^2-1}}{\Gamma(\alpha^2) 5^{\alpha^2} e^\frac{x}{5}}$ - PDF for our rvs, x - observed data

Likelihood: $L(x_1, ..., x_n;\alpha^2) = \prod_{i=1}^n  f_{\xi_i}(x_i;\alpha^2) = \prod_{i=1}^n  \frac{x_i^{\alpha^2-1}}{\Gamma(\alpha^2) 5^{\alpha^2} e^\frac{x_i}{5}} =  \frac{\prod_{i=1}^n x_i^{\alpha^2-1}}{(\Gamma(\alpha^2) 5^{\alpha^2})^n \prod_{i=1}^n e^\frac{x_i}{5}} $

Let's write down log-likelihood for more convenient maximization:

$lnL(x_1, ..., x_n;\alpha^2) =  (\alpha^2-1) \sum_i lnx_i - n ln\Gamma(\alpha^2) - n \alpha^2 ln5 - 1/5 \sum_i x_i$

Take derivative and set it to zero:

$\frac{d}{d\alpha^2}lnL(x_1, ..., x_n;\alpha^2) = -n \digamma (\alpha^2) - n ln5 + \sum_i ln(x_i)$, $\digamma$ - digamma function

$ \implies \hat{\alpha}_{ML}$ - solution of the following equation $ \digamma(\hat{\alpha}_{ML}^2) = \frac{1}{n} \sum_i ln(x_i) - ln(5)$

Analytically we cant find MLE due to digamma function, but using numerical methods like scipy.optimize.fsolve we can.

Will the estimate be unbised? \
In general, not always. MLE is asymptotically unbised.

Will the estimate be consistent? \
Yes, implies from satisfied regularity conditions.

In [10]:
import numpy as np
np.random.seed(666)
from scipy.special import digamma
from scipy.optimize import fsolve

alpha     = 4
biases    = []
variances = []

for N in [10**2, 10**3, 10**4, 10**5, 10**6, 10**7]:
    alpha_mle_list = []
    for _ in range(100):
        samples = np.random.gamma(alpha*alpha, 5, size=N)
        def equation(alpha):
            return np.mean(np.log(samples)) - np.log(5) - digamma(alpha*alpha)
        alpha_mle = fsolve(equation, x0=1)
        alpha_mle_list.append(alpha_mle)

    bias = np.mean(alpha_mle_list) - alpha
    biases.append(bias)

    variance = np.var(alpha_mle_list)
    variances.append(variance)


biases, variances

([np.float64(-0.009709897847963056),
  np.float64(-0.0001026162304582634),
  np.float64(0.0001736399033100966),
  np.float64(9.191378295536623e-05),
  np.float64(-3.0425227314889725e-05),
  np.float64(-2.671916690921705e-05)],
 [np.float64(0.002619329613532294),
  np.float64(0.00029182475613941534),
  np.float64(2.976190826684654e-05),
  np.float64(2.251346700584182e-06),
  np.float64(2.620076928004592e-07),
  np.float64(2.7349744990409028e-08)])

As in the previous case, we can conclude from the simulations that the estimate is asymptotically unbiased and consistent (this also follows from the asymptotic properties of MLE).

By the way, we can see that for MLE we have a slightly (very slightly) lower variance and bias.

### Quantile Based Estimations (min/max/median/quartile) (QB)

Like in Method of Moments in Quantile Based estimation we want to <b>match true and sample quantiles</b>.

Let's use median as most robust choice.

So, we have the following equality of sample median and true median:

$ \hat{M} = Q(0.5,\alpha^2,5), Q$ - inverse of $\Gamma(\alpha^2,5)$ CDF, $ \hat{M} $ - sample median.

There is no analytical solution here - numerical methods must be used.

$ \implies \hat{\alpha}_{QB}$ - solution of the equation above.

In [11]:
import numpy as np
np.random.seed(666)
from scipy.stats import gamma

alpha     = 4
biases    = []
variances = []

for N in [10**2, 10**3, 10**4, 10**5, 10**6, 10**7]:
    alpha_qb_list = []
    for _ in range(100):
        samples = np.random.gamma(alpha*alpha, 5, size=N)
        median = np.quantile(samples, 0.5)
        def equation(alpha):
            return gamma.ppf(0.5, a=alpha**2, scale=5) - median
        alpha_qb = fsolve(equation, x0=1)
        alpha_qb_list.append(alpha_qb)

    bias = np.mean(alpha_qb_list) - alpha
    biases.append(bias)

    variance = np.var(alpha_qb_list)
    variances.append(variance)


biases, variances

([np.float64(-0.010852099669629744),
  np.float64(0.0009758188109607602),
  np.float64(-4.78627114652852e-06),
  np.float64(0.00016108828505334571),
  np.float64(1.3972824913111026e-05),
  np.float64(-3.531876116635857e-05)],
 [np.float64(0.004122133861293637),
  np.float64(0.00046527951788792805),
  np.float64(4.2708407068008104e-05),
  np.float64(3.4865367172774974e-06),
  np.float64(4.2182206354093924e-07),
  np.float64(4.655956352061444e-08)])

As in previous cases, we have asymptotic unbiasedness and consistency.

Compared to other estimates, this estimate is significantly worse (it has twice the variance and bias).

### Combination of 3 above: (0.33 MoM + 0.33 MLE + 0.34 QB)

let's try ensemble of all estimators. maybe we will get something better with benefits of all. :)

In [23]:
import numpy as np
np.random.seed(666)
from scipy.special import digamma
from scipy.optimize import fsolve
from scipy.stats import gamma

alpha     = 4
biases    = []
variances = []

for N in [10**2, 10**3, 10**4, 10**5, 10**6, 10**7]:
    alpha_comb_list = []
    for _ in range(100):
        samples = np.random.gamma(alpha*alpha, 5, size=N)
        alpha_mom = np.sqrt(np.mean(samples)/5)

        def equation(alpha):
            return np.mean(np.log(samples)) - np.log(5) - digamma(alpha*alpha)
        alpha_mle = fsolve(equation, x0=1)

        median = np.quantile(samples, 0.5)
        def equation(alpha):
            return gamma.ppf(0.5, a=alpha**2, scale=5) - median
        alpha_qb =fsolve(equation, x0=1)

        alpha_comb = 0.33 * alpha_mom + 0.33 * alpha_mle + 0.34 * alpha_qb
        alpha_comb_list.append(alpha_comb)

    bias = np.mean(alpha_comb_list) - alpha
    biases.append(bias)

    variance = np.var(alpha_comb_list)
    variances.append(variance)


biases, variances

([np.float64(-0.010424649159400445),
  np.float64(0.0002164055145161825),
  np.float64(0.00014386415962253096),
  np.float64(0.00010842743499228646),
  np.float64(-1.7682373035565746e-05),
  np.float64(-3.0080200847937277e-05)],
 [np.float64(0.002782474426519838),
  np.float64(0.00031604459099959933),
  np.float64(3.218439112770864e-05),
  np.float64(2.3792652449613514e-06),
  np.float64(2.90698244221543e-07),
  np.float64(3.1184236737225296e-08)])

Sadly, no improvements :(((

The variance and bias are larger than in the usual MLE, and the estimation properties are the same as before.

MLE is the best!!!

Theoretically, MLE is an efficient estimate, and practically, it has the lowest variance.